<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/advanced/02_kubernetes_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kubernetes for ML Workloads

Deploy, scale, and orchestrate ML services on Kubernetes — from basic deployments to GPU workloads and ML pipelines.

**Topics:** K8s manifests, ML deployments, GPU node selection, ConfigMaps/Secrets, HPA, Argo Workflows

## 1. ML Deployment YAML Manifest Structure

In [ ]:
import yaml

# Production ML serving deployment manifest
ml_deployment = {
    'apiVersion': 'apps/v1',
    'kind': 'Deployment',
    'metadata': {'name': 'sentiment-model-v2', 'namespace': 'ml-serving'},
    'spec': {
        'replicas': 3,
        'selector': {'matchLabels': {'app': 'sentiment-model', 'version': 'v2'}},
        'strategy': {
            'type': 'RollingUpdate',
            'rollingUpdate': {'maxUnavailable': 1, 'maxSurge': 1},
        },
        'template': {
            'metadata': {'labels': {'app': 'sentiment-model', 'version': 'v2'}},
            'spec': {
                'containers': [{
                    'name': 'model-server',
                    'image': 'myregistry/sentiment-model:v2.1.0',
                    'ports': [{'containerPort': 8080}],
                    'resources': {
                        'requests': {'memory': '2Gi', 'cpu': '500m'},
                        'limits': {'memory': '4Gi', 'cpu': '2000m'},
                    },
                    'readinessProbe': {
                        'httpGet': {'path': '/health', 'port': 8080},
                        'initialDelaySeconds': 30,  # model loading time
                        'periodSeconds': 10,
                    },
                    'livenessProbe': {
                        'httpGet': {'path': '/health', 'port': 8080},
                        'initialDelaySeconds': 60,
                        'periodSeconds': 30,
                    },
                    'envFrom': [{'configMapRef': {'name': 'model-config'}}],
                    'env': [{'name': 'API_KEY', 'valueFrom': {'secretKeyRef': {'name': 'model-secrets', 'key': 'api_key'}}}],
                }],
            },
        },
    },
}

print('ML Deployment manifest (key sections):')
print(yaml.dump(ml_deployment, default_flow_style=False)[:800])

## 2. GPU Resource Requests and Node Affinity

In [ ]:
gpu_deployment = {
    'apiVersion': 'apps/v1',
    'kind': 'Deployment',
    'metadata': {'name': 'llm-inference'},
    'spec': {
        'replicas': 1,
        'template': {
            'spec': {
                # Node affinity: only schedule on GPU nodes
                'affinity': {
                    'nodeAffinity': {
                        'requiredDuringSchedulingIgnoredDuringExecution': {
                            'nodeSelectorTerms': [{
                                'matchExpressions': [
                                    {'key': 'accelerator', 'operator': 'In', 'values': ['nvidia-a100', 'nvidia-h100']},
                                ]
                            }]
                        }
                    }
                },
                # Toleration: allow scheduling on tainted GPU nodes
                'tolerations': [{'key': 'nvidia.com/gpu', 'operator': 'Exists', 'effect': 'NoSchedule'}],
                'containers': [{
                    'name': 'llm-server',
                    'image': 'myregistry/llm-server:latest',
                    'resources': {
                        'requests': {'memory': '40Gi', 'cpu': '4', 'nvidia.com/gpu': '1'},
                        'limits':   {'memory': '80Gi', 'cpu': '8', 'nvidia.com/gpu': '1'},
                        # GPU requests must equal limits (K8s GPU scheduling constraint)
                    },
                    'env': [{'name': 'CUDA_VISIBLE_DEVICES', 'value': '0'}],
                }],
            }
        }
    }
}

print('GPU scheduling in K8s:')
rules = [
    ('Node taint', 'nvidia.com/gpu=NoSchedule prevents non-GPU pods on GPU nodes'),
    ('Toleration', 'Pod must explicitly tolerate the taint to be scheduled'),
    ('Node affinity', 'Select specific GPU types (A100 vs H100 vs V100)'),
    ('Resource request', 'nvidia.com/gpu: 1 — K8s prevents GPU over-subscription'),
    ('requests == limits', 'GPU resources must match (K8s requirement)'),
]
for concept, detail in rules:
    print(f'  {concept:<20}: {detail}')

## 3. ConfigMaps and Secrets for ML Configs

In [ ]:
import subprocess

# ConfigMap: non-sensitive model config
configmap = """
apiVersion: v1
kind: ConfigMap
metadata:
  name: model-config
  namespace: ml-serving
data:
  MODEL_NAME: "distilbert-sentiment-v2"
  MAX_BATCH_SIZE: "32"
  MAX_SEQUENCE_LENGTH: "512"
  LOG_LEVEL: "INFO"
  ENABLE_CACHING: "true"
"""

# Secret: sensitive credentials (base64 encoded in real K8s)
secret_manifest = """
apiVersion: v1
kind: Secret
metadata:
  name: model-secrets
  namespace: ml-serving
type: Opaque
stringData:  # K8s auto-encodes to base64
  openai_api_key: "sk-..."
  mlflow_tracking_uri: "http://mlflow:5000"
  redis_password: "super-secret"
"""

# In Python: generate ConfigMap from a config dict
def dict_to_configmap(name: str, namespace: str, config: dict) -> dict:
    return {
        'apiVersion': 'v1',
        'kind': 'ConfigMap',
        'metadata': {'name': name, 'namespace': namespace},
        'data': {k: str(v) for k, v in config.items()},
    }

model_config = {'MODEL_NAME': 'bert-v3', 'MAX_BATCH_SIZE': 64, 'LOG_LEVEL': 'INFO'}
cm = dict_to_configmap('model-config', 'ml-serving', model_config)

print('Generated ConfigMap:')
print(yaml.dump(cm))
print('Best practice: Never put API keys in ConfigMaps — use Secrets or External Secrets Operator.')

## 4. HorizontalPodAutoscaler for Inference Services

In [ ]:
# HPA scales pods based on CPU, memory, or custom metrics
hpa_manifest = {
    'apiVersion': 'autoscaling/v2',
    'kind': 'HorizontalPodAutoscaler',
    'metadata': {'name': 'sentiment-model-hpa'},
    'spec': {
        'scaleTargetRef': {
            'apiVersion': 'apps/v1',
            'kind': 'Deployment',
            'name': 'sentiment-model-v2',
        },
        'minReplicas': 2,
        'maxReplicas': 10,
        'metrics': [
            {
                'type': 'Resource',
                'resource': {
                    'name': 'cpu',
                    'target': {'type': 'Utilization', 'averageUtilization': 70},
                },
            },
            {
                # Custom metric: requests per second from Prometheus
                'type': 'Pods',
                'pods': {
                    'metric': {'name': 'http_requests_per_second'},
                    'target': {'type': 'AverageValue', 'averageValue': '100'},
                },
            },
        ],
        'behavior': {
            'scaleUp': {'stabilizationWindowSeconds': 30},  # scale up fast
            'scaleDown': {'stabilizationWindowSeconds': 300},  # scale down slow (avoid flapping)
        },
    },
}

print('HPA manifest:')
print(yaml.dump(hpa_manifest, default_flow_style=False))
print('Scaling behavior:')
print('  scaleUp stabilization=30s: waits 30s before scaling up (absorb spikes)')
print('  scaleDown stabilization=300s: waits 5min before scaling down (avoid thrashing)')

## 5. Argo Workflows for ML Pipelines

In [ ]:
from hera.workflows import DAG, Task, Workflow, Steps, Step
from hera.workflows import script

# Define pipeline steps as Python functions
@script(image='python:3.11', resources={'memory': '2Gi', 'cpu': '1'})
def fetch_data(dataset_uri: str) -> str:
    import subprocess
    print(f'Fetching dataset from {dataset_uri}')
    # In real pipeline: download from S3, GCS, etc.
    return '/data/dataset.parquet'

@script(image='python:3.11', resources={'memory': '4Gi', 'cpu': '2'})
def preprocess(data_path: str, output_path: str) -> str:
    print(f'Preprocessing {data_path} → {output_path}')
    return output_path

@script(image='pytorch/pytorch:2.1-cuda12.1-cudnn8-runtime',
        resources={'memory': '16Gi', 'cpu': '4', 'nvidia.com/gpu': '1'})
def train_model(data_path: str, model_output: str, epochs: int = 10) -> str:
    print(f'Training for {epochs} epochs on {data_path}')
    return model_output

@script(image='python:3.11', resources={'memory': '2Gi', 'cpu': '1'})
def evaluate_and_register(model_path: str, threshold: float = 0.90):
    print(f'Evaluating {model_path}, threshold={threshold}')
    # Register to MLflow if accuracy > threshold

# Equivalent YAML workflow (what Argo actually runs)
workflow_yaml = """
apiVersion: argoproj.io/v1alpha1
kind: Workflow
metadata:
  name: ml-training-pipeline
spec:
  entrypoint: pipeline
  templates:
  - name: pipeline
    dag:
      tasks:
      - name: fetch-data
        template: fetch-data-template
      - name: preprocess
        dependencies: [fetch-data]
        template: preprocess-template
      - name: train
        dependencies: [preprocess]
        template: train-template
      - name: evaluate
        dependencies: [train]
        template: evaluate-template
"""

print('Argo Workflows ML pipeline structure:')
print('  fetch-data → preprocess → train (GPU) → evaluate & register')
print()
print('Key advantages over Airflow for ML:')
print('  - Each step runs in its own container (dependency isolation)')
print('  - Native GPU support via K8s resource requests')
print('  - Artifact passing between steps via S3/GCS')
print('  - Built-in retry, caching, and parallelism')

## Exercises

1. **Rolling update safety:** Write a script that monitors a K8s deployment rollout (`kubectl rollout status`) and automatically rolls back (`kubectl rollout undo`) if the error rate from Prometheus exceeds 5% within 2 minutes of deployment.

2. **Custom HPA metric:** Implement a Flask endpoint that exposes a custom metric (inference queue depth) in Prometheus format. Write the K8s HPA manifest that scales the deployment based on this metric.

3. **Multi-step Argo pipeline:** Create a complete Argo Workflow that: (1) downloads a dataset, (2) runs preprocessing in parallel on 3 shards, (3) merges results, (4) trains a model, (5) evaluates it, and (6) conditionally registers to MLflow only if accuracy > 90%.